# M3 · Give records a shape and state an owner

Records name related facts; instances own changes; snapshots retain history. Build those responsibilities one step at a time.

**Start:** Use a Python 3 kernel; these cells use only the standard library. Restart the kernel and run top to bottom. All teaching Python is in the editable cells below; no repository imports or external lesson loaders are needed.

**Explore:** Predict → run → observe → change one input. Expected results assume the supplied values and a fresh run; changing earlier state may change later results. To return to the baseline, restart and rerun from the start.

## Exploration 1 — build and observe the mechanism

### 1. Give a product a named record shape

A record groups related facts under meaningful field names. A dataclass supplies routine record behavior.

**Before running:** Predict the result or state change.

In [ ]:
from dataclasses import dataclass, field, FrozenInstanceError

@dataclass(frozen=True)
class Product:
    product_id: str
    name: str
    price_paise: int

cable = Product("1", "USB-C Cable", 49900)
print(cable)


**Change and explain:** Construct the bottle record with price 89900.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The representation includes the named fields and their values.

</details>

### 2. Separate equality from identity

Equal records can be different objects. Hints describe intended values but do not themselves validate construction.

**Before running:** Predict the result or state change.

In [ ]:
same_cable = Product("1", "USB-C Cable", 49900)
print(cable == same_cable, cable is same_cable)
invalid_product = Product("1", "USB-C Cable", "expensive")
print(type(invalid_product.price_paise).__name__)


**Change and explain:** Explain which later boundary mechanism validates actual types.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Equality is True, identity False; the dataclass accepts the string price.

</details>

### 3. Observe and recover from forbidden reassignment

Frozen records prevent field reassignment. They do not promise that arbitrary nested objects become immutable.

**Before running:** Predict the result or state change.

In [ ]:
try:
    cable.price_paise = 100
except FrozenInstanceError as error:
    print(type(error).__name__)
revised_cable = Product(cable.product_id, cable.name, 50000)
print(cable.price_paise, revised_cable.price_paise)


**Change and explain:** Explain why creating a new record preserves the old fact.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The original price stays 49900; the new record has 50000.

</details>

### 4. Give changing quantities an owner

An instance owns its state. A method applies a rule to the instance receiving the call.

**Before running:** Predict the result or state change.

In [ ]:
class ShoppingCart:
    def __init__(self):
        self.quantities = {}

    def add(self, product_id, quantity=1):
        if type(quantity) is not int or quantity <= 0:
            raise ValueError("Quantity must be a positive integer")
        self.quantities[product_id] = self.quantities.get(product_id, 0) + quantity

cart = ShoppingCart()
cart.add("1", 2)
print(cart.quantities)


**Change and explain:** Add one more cable using the default argument.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

This cart owns {"1": 2}; the tiny example focuses on quantity ownership, with catalog lookup in checkout.

</details>

### 5. Compare two independent owners

The same class can describe many independent instances. Changes belong to the chosen receiver.

**Before running:** Predict the result or state change.

In [ ]:
other_cart = ShoppingCart()
other_cart.add("3")
print(cart.quantities)
print(other_cart.quantities)


**Change and explain:** Change the second cart; predict the first cart’s contents.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The first cart has cables; the second has a bottle.

</details>

### 6. Observe the shared-class-state defect

A mutable class attribute can be shared by instances. Use instance state for independent carts.

**Before running:** Predict the result or state change.

In [ ]:
class SharedCart:
    quantities = {}

first = SharedCart()
second = SharedCart()
first.quantities["1"] = 2
print(second.quantities)
print(ShoppingCart().quantities)


**Change and explain:** Locate the different lifetime of the two dictionaries.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The flawed second cart sees the cable; a correctly constructed cart starts empty.

</details>

## Exploration 2 — change a condition and inspect the consequence

### 7. Use a factory for a mutable record field

A factory creates fresh state for each dataclass instance. A literal shared object would have the wrong lifetime.

**Before running:** Predict the result or state change.

In [ ]:
@dataclass
class CartDraft:
    quantities: dict[str, int] = field(default_factory=dict)

first_draft = CartDraft()
second_draft = CartDraft()
first_draft.quantities["1"] = 2
print(first_draft.quantities, second_draft.quantities)


**Change and explain:** Create a third draft and predict its contents.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The second draft remains empty.

</details>

### 8. Retain facts in a new line record

Checkout is a historical boundary. Copy the scalar facts that must remain true after the draft changes.

**Before running:** Predict the result or state change.

In [ ]:
@dataclass(frozen=True)
class OrderLine:
    product_id: str
    name: str
    unit_price_paise: int
    quantity: int

line = OrderLine(cable.product_id, cable.name, cable.price_paise, 2)
print(line.quantity, line.unit_price_paise)


**Change and explain:** Explain why an order needs a unit-price snapshot as well as a product ID.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The line retains two units at 49900 paise each.

</details>

### 9. Compose records and derive a total

Composition connects records. Computing a total from lines avoids a second independently mutable total.

**Before running:** Predict the result or state change.

In [ ]:
@dataclass(frozen=True)
class Order:
    order_id: str
    lines: tuple[OrderLine, ...]

    @property
    def total_paise(self):
        return sum(item.unit_price_paise * item.quantity for item in self.lines)

order = Order("demo-order", (line,))
print(order.total_paise)


**Change and explain:** Add another immutable line and predict the new total.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The one-line order totals 99800 paise.

</details>

### 10. Construct a snapshot from the owned draft

Ordinary functions can compose records without owning mutable state. Copy before clearing the successful draft.

**Before running:** Predict the result or state change.

In [ ]:
from uuid import uuid4
catalog = {"1": cable, "3": Product("3", "Water Bottle", 89900)}

def checkout(cart):
    if not cart.quantities:
        raise ValueError("Cannot checkout an empty cart")
    lines = tuple(OrderLine(key, catalog[key].name, catalog[key].price_paise, count)
                  for key, count in cart.quantities.items())
    order = Order(uuid4().hex, lines)
    cart.quantities.clear()
    return order

saved_order = checkout(cart)
print(saved_order.total_paise, cart.quantities)


**Change and explain:** If lookup fails, does clearing happen?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Successful checkout empties the cart; a failure before clearing preserves it.

</details>

### 11. Change current state after checkout

Independent snapshots remain historical even when today’s state changes.

**Before running:** Predict the result or state change.

In [ ]:
cart.add("1", 5)
catalog["1"] = Product("1", "USB-C Cable", 99900)
print(cart.quantities)
print(saved_order.lines[0].quantity)
print(saved_order.lines[0].unit_price_paise)
print(saved_order.total_paise)


**Change and explain:** Explain which object owns each displayed quantity and price.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Current cart quantity is 5; the saved quantity is 2 and the saved total remains 99800.

</details>

### 12. See why a tuple alone is insufficient

A fixed outer container can still refer to changing inner objects. Snapshot independence comes from its contents too.

**Before running:** Predict the result or state change.

In [ ]:
mutable_item = {"product_id": "1", "quantity": 2}
unsafe_snapshot = (mutable_item,)
mutable_item["quantity"] = 5
print(unsafe_snapshot[0]["quantity"])
print(saved_order.lines[0].quantity)


**Change and explain:** In the modular Pico-Zon, locate records, cart, checkout, and CLI; explain why they have different responsibilities.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The unsafe snapshot changes to 5; the independent record remains 2.

</details>

## Apply the mechanism in Nano-Zon

Follow the [M3 lab guide](../../projects/nano-zon/labs/m03/README.md). The codealong isolates the mechanism; the lab combines it with the supplied application.

State the principle, point to the code that expresses it, then explain one limit of the observed result.

### Connect to modules

Open the [modular Pico-Zon guide](../../projects/pico-zon/modular/README.md). Locate models, catalog, cart, checkout, and CLI; compare their responsibilities with the records, functions, and objects you just executed. Local-module organization is demonstrated in that project; the notebook stays self-contained.

**Authoring status:** This notebook contains editable teaching source, not captured execution output. Syntax and structure can be inspected independently of execution; runtime delivery review remains pending.